# Cluster Sentences: blablabla

This notebook looks into techniques for defining clusters.
- First, sentences duplicates with a cosine similarity score of >=1 are removed.
- Compare resulting clusters of two lemmatisation methods (nltk, spacy, no lemma).

### Settings

In [ ]:
# Settings
embedding_file = "embedding_corpus_free_3600_250606.pickle"
similarity_file = "similarity_corpus_free_3600_250606.pickle"

raw_corpus_file = "corpus_free_3600_250606.csv"
used_st_model = "NeuML/pubmedbert-base-embeddings"

# files for comparing lemmatisation methods
# Multiple vector files
folder = "explore_vectors"
filename1 = 'no_lemma_applied_df_xml2corpus_by_sentence_pmid_free_3600_250606.pickle'
filename2 = 'nltk_df_xml2corpus_by_sentence_pmid_free_3600_250606.pickle'
filename3 = 'spacy_df_xml2corpus_by_sentence_pmid_free_3600_250606.pickle'

## Initialisation

### Import and functions

In [60]:
# import
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
from sentence_transformers import SentenceTransformer

### Functions

In [46]:
def get_duplets_dataframe(similarity_matrix, threshold: float = 1.0):
    """Return a similarity dataframe that only contains duplets, where duplets are defined as sentences with a similarity score equal or higher than provided threshold."""
    # Substract 2 from the cosine similarity score that matched itself, so that it turns the value to -1.
    df_duplets = pd.DataFrame(
        np.subtract(
            similarity_matrix, 
            np.identity(similarity_matrix.shape[0]) *2
        )
    )

    # Drop columns and rows without duplets (defined by the threshold)
    df_duplets = df_duplets[df_duplets >= threshold].dropna(axis=0, how='all')
    df_duplets = df_duplets[df_duplets >= threshold].dropna(axis=1, how='all')

    # if df_duplets is empty (and therefore there are no duplets)
    if len(df_duplets.index) == 0:
        # raise an error
        raise ValueError("No duplets found in provided pd.DataFrame!")

    return df_duplets

def get_duplets_clusters(df_duplets):
    """Return a dictionary where the kept 'unique sentence index' (=key) references to the duplet indice in a list (=values)."""
    # Create a dict with indices on sentences with a duplet.
    potential_duplets = df_duplets.index
    duplet_dict = dict()

    # Iterate over potential duplets list
    # TODO: There should be a nicer way with groupby or something
    while len(potential_duplets) != 0:
        
        # Look at first item
        idx = potential_duplets[0]

        # For idx in list, find other idx that have same sentence.
        cluster = df_duplets.loc[idx, ~df_duplets.loc[idx].isna()].index

        # Store 'unique' idx in as key, and store 'duplet' idx as values
        duplet_dict[idx] = cluster.to_list()

        # Remove duplet indice and first item from potential_duplets. (This will shorten the loop)
        potential_duplets = potential_duplets[~potential_duplets.isin(cluster)]
        potential_duplets = potential_duplets[1:]

    return duplet_dict

### Load

In [56]:
# Load raw corpus
df_corpus = pd.read_csv(f'../../data/corpus/{raw_corpus_file}')

# Load cosine similarity scores
with open(f"../../data/vectors/{similarity_file}", 'rb') as handle:
    similarities = pickle.load(handle)

# Load sentence embeddings
with open(f"../../data/vectors/{embedding_file}", 'rb') as handle:
    embeddings = pickle.load(handle)


In [48]:
# sample
similarities[:5, :5]

tensor([[1.0000, 0.1074, 0.2322, 0.5048, 0.3413],
        [0.1074, 1.0000, 0.4907, 0.3195, 0.5344],
        [0.2322, 0.4907, 1.0000, 0.3907, 0.4358],
        [0.5048, 0.3195, 0.3907, 1.0000, 0.3484],
        [0.3413, 0.5344, 0.4358, 0.3484, 1.0000]])

Similarity scores are correctly loaded if row and column indices with the same number == `1`

In [49]:
embeddings.shape

(18305, 768)

In [50]:
similarities.shape

torch.Size([18305, 18305])

## Remove duplets (So that one unique remains)

As noted in notebook `sentence_pairs.ipynb`, there are multiple duplicated sentences. 
For clustering I would like to remove these duplicates, since we are interested in sentence similarities, I think that the 'high similarity' duplicate sentences have may influence the formation of clusters.

In [ ]:
# Visualize clusters of duplets
sns.clustermap(get_duplets_dataframe(similarities).fillna(0))

First occurrence is saved. The rest will be removed. (This will automatically shorten the list)
I cannot think of a reason why it would matter which sentence would be removed.

In [ ]:
# get duplet dictionary
duplet_dict = get_duplets_clusters(
    get_duplets_dataframe(
        similarities
    )
)

# Create a list with indexes on sentences with a duplet.
duplet_to_remove = [item for layer in duplet_dict.values() for item in layer]

# Drop duplets from `embeddings` and `similarity`
print(f"embedding shape with duplets: {embeddings.shape}")
embeddings = np.delete(embeddings, duplet_to_remove, 0)
print(f"embedding shape without duplets: {embeddings.shape}")

# TODO: there is probably an elegant way with torch, 
# but after a lot of time I did not manage to figure it out...
print(f"similarities shape with duplets: {similarities.shape}")
similarities = pd.DataFrame(similarities).drop(duplet_to_remove, axis= 0)
similarities = similarities.drop(duplet_to_remove, axis= 1)
print(f"similarities shape without duplets: {similarities.shape}")

embedding shape with duplets: (18305, 768)
embedding shape without duplets: (18270, 768)
similarities shape with duplets: torch.Size([18305, 18305])
similarities shape without duplets: (18270, 18270)


In [59]:
# Check if duplets are gone:
try:
    get_duplets_dataframe(
        similarities
    )

except ValueError:
    print("No duplets found in matrix.")

No duplets found in matrix


In [63]:
# Save `duplet_dict` as .csv
# (This is important, because if the 'duplet' is a hit, we need to find it back in the corpus. Also if the hit occurs multiple times in the corpus.)

# Create `vectors` directory, if it does not yet exists.
Path("../../data/vectors/duplets").mkdir(exist_ok = True)

# Save 
with open(f"../../data/vectors/duplets/dict_{'_'.join(embedding_file.split('_')[2:])}", "wb") as handle:
    pickle.dump(embeddings, handle)

## Test

In [ ]:
import torch

torch.BoolTensor()

# torch.masked_select(similarities, ~cluster)